# ECM generation: beginning of life

`generate_ecm` fits an equivalent circuit model (ECM) to the physics model of the exact design you are working on, in two stages:

1. **Pulse fit.** The physics model runs a virtual pulse characterisation, a pulse followed by a rest at every point of the SoC, temperature and C-rate grid (60 s and 30 min by default). An RC network is fitted to each pulse and its relaxation, which gives R0 and the RC pairs.
2. **Constant current refit.** A pulse does not show the resistance a cell builds up over a long charge or discharge. The physics model also runs a constant current charge and discharge at every temperature and C-rate of the grid, and the slowest RC pair is refitted to match them.

In a MATLAB parameter set downloaded with `save_mat` (see *Download a MATLAB parameter set* below), `ecmParams.meta.modelParameterisationVersion` ends in `_v5` when both stages ran, for example `ECM_2RC_v5`. `ecmParams.meta.modelVersion` is the version of the Breathe Simulate TECM / ECM blocks the file is built for (`v1.0.0`).

This notebook fits the ECM of a fresh cell, at beginning of life (BOL), and exports it to MATLAB. Notebook *14b ECM Generation Aged Cells* fits the ECM of the same cell after a degradation campaign.

## A fresh ECM for a design

`ecm_options` sets up the fit. A `run_sim` style dict passed as `design=...` characterises a modified cell instead of the library one.

The grid below is the one used for the 2nd order CATL LFP ECM: SoC breakpoints are dense below 0.1 and above 0.8, where the flat LFP OCV curve bends sharply, temperatures run from 0 to 50 degC in 10 degC steps, and the same four C-rates are used in both directions.

In [1]:
from breathe_simulate import api_interface as api
from breathe_simulate.ecm import ecm_options

cell_name = "Molicel P45B"

options = ecm_options(
    n_rc=2,
    soc_grid=[
        0.0,
        0.02,
        0.05,
        0.1,
        0.2,
        0.3,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.85,
        0.9,
        0.95,
        0.98,
        1.0,
    ],
    temp_grid_degC=[0.0, 10.0, 20.0, 30.0, 40.0, 50.0],
    charge_c_rates=[0.5, 1.0, 2.0, 3.0],
    discharge_c_rates=[0.5, 1.0, 2.0, 3.0],
)

fresh = api.generate_ecm(cell_name, ecm=options)
fresh

Submitted run_sim job 31fee471-fd10-484f-97ba-ad43d48fb017


EcmResults(2RC, 16 SoC x 6 T x 4 dch / 4 chg rates)

## The parameter tables

`to_dataframe()` flattens the fitted tables into one long table.

In [2]:
table = fresh.to_dataframe()
table.head(12)

,soc,temperature_degC,c_rate,direction,R0_Ohm,R1_Ohm,C1_F,R2_Ohm,C2_F
0,0.0,0.0,0.5,discharge,0.011520,0.017148,1563.819543,0.061071,4073.091167
1,0.0,0.0,1.0,discharge,0.011684,0.018766,1321.724038,0.048065,4919.180451
2,0.0,0.0,2.0,discharge,0.011448,0.027170,641.676432,0.037461,5349.013588
3,0.0,0.0,3.0,discharge,0.011448,0.027170,641.676432,0.038107,5258.406009
4,0.0,10.0,0.5,discharge,0.009777,0.010794,1744.569438,0.028610,5885.334467
5,0.0,10.0,1.0,discharge,0.009971,0.010867,1629.679219,0.022517,7163.632255
6,0.0,10.0,2.0,discharge,0.009951,0.012769,1209.057925,0.017549,8176.716983
7,0.0,10.0,3.0,discharge,0.009951,0.012769,1209.057925,0.017852,8038.210599
8,0.0,20.0,0.5,discharge,0.008415,0.007594,1840.871961,0.014591,8454.755561
9,0.0,20.0,1.0,discharge,0.008587,0.007226,1737.558369,0.011483,9991.098406


`r0`, `r(k)` and `c(k)` return the raw 3D arrays over (SoC x temperature x C rate).

In [3]:
print("R0 discharge table shape:", fresh.r0("discharge").shape)
print("SoC grid:", fresh.soc_grid)
print("Temperature grid [degC]:", fresh.temp_grid_degC)
print("Capacity [Ah]:", fresh.capacity)

fresh.plot_parameters("R0", direction="discharge", c_rate=1.0)

R0 discharge table shape: (16, 6, 4)
SoC grid: [0.   0.02 0.05 0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  0.85 0.9  0.95
 0.98 1.  ]
Temperature grid [degC]: [ 0. 10. 20. 30. 40. 50.]
Capacity [Ah]: 4.4999999999999964


Each direction and each C rate gets its own table. Comparing them shows the asymmetry between charge and discharge pulses and how the resistance shifts with current.

In [4]:
import numpy as np
import plotly.graph_objects as go

temp_index = int(np.argmin(np.abs(fresh.temp_grid_degC - 20.0)))  # 20 degC

fig = go.Figure()
for direction, rates in (
    ("charge", fresh.charge_c_rates),
    ("discharge", fresh.discharge_c_rates),
):
    for i, rate in enumerate(rates):
        fig.add_trace(
            go.Scatter(
                x=fresh.soc_grid,
                y=1000 * fresh.r0(direction)[:, temp_index, i],
                mode="lines+markers",
                name=f"{rate:g}C {direction}",
            )
        )
fig.update_layout(
    title="R0 vs SoC at 20 degC, both directions and pulse currents",
    xaxis_title="SoC",
    yaxis_title="R0 [mOhm]",
    legend_title="Pulse",
)
fig

## OCV, thermal constants and saving

The result also carries the OCV curves and thermal constants an ECM runtime needs, so it is a complete model on its own. The charge and discharge branches of the OCV show the hysteresis the mean curve averages out.

In [5]:
fresh.ocv.head()

,soc,mean_voltage_V,charge_voltage_V,discharge_voltage_V,hysteresis_voltage_V
0,0.000,2.663782,2.827584,2.499981,0.163802
1,0.001,2.685911,2.851792,2.520031,0.165880
2,0.002,2.705992,2.872798,2.539186,0.166806
3,0.003,2.724833,2.891929,2.557738,0.167096
4,0.004,2.742614,2.909565,2.575664,0.166951


In [6]:
fig = go.Figure()
for column, label in (
    ("charge_voltage_V", "charge branch"),
    ("discharge_voltage_V", "discharge branch"),
    ("mean_voltage_V", "mean"),
):
    fig.add_trace(
        go.Scatter(x=fresh.ocv["soc"], y=fresh.ocv[column], mode="lines", name=label)
    )
fig.update_layout(
    title="Open circuit voltage and its hysteresis",
    xaxis_title="SoC",
    yaxis_title="Voltage [V]",
    legend_title="Curve",
)
fig

In [7]:
{
    key: f"curve with {len(value)} points" if isinstance(value, list) else value
    for key, value in fresh.thermal.items()
}

{'dudt_VK': 'curve with 1001 points',
 'soc_dudt': 'curve with 1001 points',
 'cell_mass_kg': 0.06822167545418895,
 'cell_surface_area_m2': 0.005370967488621659,
 'cell_heat_capacity_J_kgK': 1200.0,
 'thermal_conductivity_in_w_mk': 42.23552620603363,
 'thermal_conductivity_through_w_mk': 1.0855399470076916}

In [8]:
from breathe_simulate.results import EcmResults

fresh.save("fresh_ecm.json")
reloaded = EcmResults.load("fresh_ecm.json")
reloaded

EcmResults(2RC, 16 SoC x 6 T x 4 dch / 4 chg rates)

## Download a MATLAB parameter set

`save_mat` writes the fitted ECM as a `.mat` file for the Breathe Simulate 1st, 2nd and 3rd order ECM blocks in Simulink. The file holds one struct, `ecmParams`, with:

- `meta`: cell name, voltage limits, nominal capacity and energy, mass and surface area
- `ocv`: the mean, charge and discharge OCV curves and the hysteresis constants
- `charge_LUTs` / `discharge_LUTs`: R0 and R, tau and C for each RC pair over SoC x temperature x C rate
- `modelOrder`: the number of RC pairs
- `thermal`: heat capacity, the entropic coefficient dU/dT and the thermal conductivities

Discharge C rates are stored as negative breakpoints, following the Breathe sign convention.

In [9]:
mat_path = "Molicel_P45B_ecmParams_2RC.mat"
fresh.save_mat(mat_path, cell_name=cell_name)

Loading the file back with SciPy shows what MATLAB will see.

In [10]:
from scipy.io import loadmat

params = loadmat(mat_path, simplify_cells=True)["ecmParams"]
meta = params["meta"]
discharge_luts = params["discharge_LUTs"]

print("Fields:", list(params))
print(
    f"{meta['cellName']}: {params['modelOrder']} RC pairs, "
    f"{meta['nominalCap_Ah']:.2f} Ah, {meta['voltMin_V']:.2f} to {meta['voltMax_V']:.2f} V"
)
print("Discharge R0 table (SoC x T x C rate):", discharge_luts["R0_Ohm"].shape)
print("SoC breakpoints:", discharge_luts["socBPs_dim1"])
print("Temperature breakpoints [degC]:", discharge_luts["tempBPs_degC_dim2"])
print("Discharge C-rate breakpoints:", discharge_luts["currentBPs_CRate_dim3"])

Fields: ['meta', 'ocv', 'charge_LUTs', 'discharge_LUTs', 'modelOrder', 'thermal']
Molicel P45B: 2 RC pairs, 4.50 Ah, 2.50 to 4.20 V
Discharge R0 table (SoC x T x C rate): (16, 6, 4)
SoC breakpoints: [0.   0.02 0.05 0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  0.85 0.9  0.95
 0.98 1.  ]
Temperature breakpoints [degC]: [ 0. 10. 20. 30. 40. 50.]
Discharge C-rate breakpoints: [-3.  -2.  -1.  -0.5]


In Jupyter, the link below downloads the file. In MATLAB, load it and point the ECM block at the struct:

```matlab
load("Molicel_P45B_ecmParams_2RC.mat")  % creates ecmParams in the workspace
% Breathe Simulate 2nd order ECM block: set the parameter p to ecmParams
```

Use the block whose order matches `modelOrder`, here the 2nd order block for `n_rc=2`.

In [11]:
from IPython.display import FileLink

FileLink(mat_path)

D:\Git\Breathe_Design_Product\breathe_simulate\docs\examples\Molicel_P45B_ecmParams_2RC.mat